# Router Architectures & Modes

In this notebook, we dive deeper into **how** the router makes decisions. We'll explore different architectures and routing modes.

**Goals:**
1. Load different router types: `Classical`, `Pairwise`, and `Reward`.
2. Compare routing modes: `accuracy`, `cheap`, `fast`, `balanced`.
3. See how the choice changes based on the mode.

---

In [1]:
# 1. Setup Paths
import sys
import os
from pathlib import Path

current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent
sys.path.append(str(project_root))

try:
    import artemis_final
    from artemis_final.router.public_api import load_router_from_checkpoint
    print(" Imports successful")
except ImportError as e:
    print(f" Import failed: {e}")

 Imports successful


In [2]:
# 2. Load Router
# We load the Multitask Reward Router which supports dynamic modes.

CHECKPOINT_PATH = project_root / "artemis_final" / "checkpoints" / "best_multitask_router_v1.pt"

router = None

if CHECKPOINT_PATH.exists():
    try:
        print(f"Loading Reward router from {CHECKPOINT_PATH.name}...")
        router = load_router_from_checkpoint(router_type="reward", checkpoint_path=str(CHECKPOINT_PATH), verbose=False)
        print(" Router Loaded")
    except Exception as e:
        print(f"️ Failed to load router: {e}")
else:
    print(f"️ Checkpoint missing at {CHECKPOINT_PATH}")


️ Checkpoint missing at /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints/best_multitask_router_v1.pt


In [3]:
# 3. Routing Modes Comparison
modes = ["accuracy", "balanced", "cheap", "fast"]
prompts = [
    "Describe the artistic style of this painting in detail.", # Complex
    "Is there a dog in this picture? Yes or no.",             # Simple
]

from PIL import Image
dummy_image = Image.new('RGB', (224, 224), color='blue')

import pandas as pd

results = []

if router:
    for prompt in prompts:
        for mode in modes:
            try:
                decision = router.route(prompt=prompt, image=dummy_image, mode=mode)
                results.append({
                    "Prompt": prompt[:30] + "...",
                    "Mode": mode,
                    "Selected Model": decision['chosen_model'],
                    "Reward": f"{decision['rewards'][decision['chosen_model']]:.4f}"
                })
            except Exception as e:
                print(f"Error in {mode} mode: {e}")

    df = pd.DataFrame(results)
    print("Routing Decisions Comparison:")
    display(df)
else:
    print("Router not available.")

Router not available.


## Visualizing the Choices

Let's see the distribution of models selected for the simple vs complex prompt.

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x="Selected Model", hue="Mode")
    plt.title="Model Selection by Mode"
    plt.show()
else:
    print("No results to plot.")

NameError: name 'df' is not defined